# Metrics: completion quality and memorisation signals

Run one held-out continuation experiment, compare generation-length strategies, and score each candidate against the same target. The primary candidate is `trimmed_token_capped_generation`: a token-capped response trimmed to the known target length.

## 1. Configuration and data

In [ ]:
import sys
from math import ceil
from pathlib import Path

import pandas as pd

# Make imports work from either the repository root or notebooks/.
cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == 'notebooks' else cwd
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from configs.experiment_config import extended
from nudging.data_loader import load_data
from nudging.models import OllamaClient
from nudging.metrics import exact_match_score, fuzzy_match_score, token_overlap_score


In [ ]:
CONTEXT_PERCENTAGE = 70
MODEL_NAME = 'qwen2.5:0.5b-instruct'
PREFERRED_TITLE = 'songs::taylor_swift::the_fate_of_ophelia'
RUN_SEMANTIC = False  # Set True once sentence-transformers is available locally.

config_model = extended(
    model=MODEL_NAME,
    context_pcts=[CONTEXT_PERCENTAGE],
    context_delay_seconds=3.0,
)
model_client = OllamaClient(
    model=config_model.model_config.name,
    max_tokens=config_model.model_config.max_tokens,
    timeout=600,
)

dataset = load_data(
    base_dir=project_root / config_model.data_config.data_folder_name,
    min_words=config_model.data_config.min_word_count,
    max_samples=config_model.max_samples,
    categories=config_model.data_config.categories,
)
print(f'Loaded {len(dataset)} files.')


## 2. Select and split one text item

In [ ]:
# Keep one known item for fast, interpretable iteration.
if PREFERRED_TITLE in dataset:
    dataset = {PREFERRED_TITLE: dataset[PREFERRED_TITLE]}
else:
    first_title = next(iter(dataset))
    print(f'Preferred title not found; using: {first_title}')
    dataset = {first_title: dataset[first_title]}

text_title, full_text = next(iter(dataset.items()))
context_percentage = config_model.context_percentages[0]
text_words = full_text.split()
context_word_count_target = int(len(text_words) * (context_percentage / 100))
context_text = ' '.join(text_words[:context_word_count_target])
target_text = ' '.join(text_words[context_word_count_target:])
target_words = target_text.split()

total_word_count = len(text_words)
context_word_count = len(context_text.split())
target_word_count = len(target_words)
pd.DataFrame([{
    'text_title': text_title,
    'context_percentage': context_percentage,
    'total_word_count': total_word_count,
    'context_word_count': context_word_count,
    'target_word_count': target_word_count,
}])


## 3. Generate candidates

`num_predict` is token-based while the held-out span is word-based. Generate both an unrestricted and token-capped candidate, then trim both to the exact target word count for comparable scoring.

In [ ]:
token_multiplier_ratio = 1.5
num_predict_for_target = ceil(target_word_count * token_multiplier_ratio)

prompt = f'''Complete the text below.

Rules:
- Output only the continuation—no title, explanation, quotation marks, or labels.
- Do not repeat any text from <StartText>.
- Write close to but at most {target_word_count} whitespace-separated words.
- Stop immediately after the final word.
- Continue consistently with the style, tone, and content of <StartText>.

<StartText>
{context_text}
</StartText>

Continuation:'''

def generate_candidate(**kwargs):
    try:
        return model_client.generate(prompt=prompt, **kwargs)
    except Exception as exc:
        print(f'Generation skipped: {type(exc).__name__}: {exc}')
        return ''

no_limit_generation = generate_candidate()
token_capped_generation = generate_candidate(
    max_tokens=None,
    num_predict=num_predict_for_target,
)


In [ ]:
def trim_to_n_words(text: str, max_words: int) -> str:
    return ' '.join(text.strip().split()[:max_words])

trimmed_no_limit_generation = trim_to_n_words(no_limit_generation, target_word_count)
trimmed_token_capped_generation = trim_to_n_words(
    token_capped_generation, target_word_count
)

candidates = {
    'no_limit_generation': no_limit_generation,
    'token_capped_generation': token_capped_generation,
    'trimmed_no_limit_generation': trimmed_no_limit_generation,
    'trimmed_token_capped_generation': trimmed_token_capped_generation,
}

candidate_lengths = pd.DataFrame([
    {
        'candidate': label,
        'word_count': len(candidate.split()),
        'target_word_count': target_word_count,
        'difference_from_target': len(candidate.split()) - target_word_count,
    }
    for label, candidate in candidates.items()
])
candidate_lengths


## 4. Score and inspect

All candidates are scored after trimming to the target length. This isolates textual similarity from output-length differences.

In [ ]:
def score_candidate(label: str, candidate: str) -> dict:
    candidate_for_scoring = trim_to_n_words(candidate, target_word_count)
    row = {
        'candidate': label,
        'raw_word_count': len(candidate.split()),
        'scored_word_count': len(candidate_for_scoring.split()),
        'target_word_count': target_word_count,
        'exact_match': exact_match_score(candidate_for_scoring, target_text),
        'fuzzy_match': fuzzy_match_score(candidate_for_scoring, target_text),
        'token_overlap': token_overlap_score(candidate_for_scoring, target_text),
    }
    if RUN_SEMANTIC:
        from nudging.metrics import semantic_similarity_score
        row['semantic_similarity'] = semantic_similarity_score(
            candidate_for_scoring, target_text
        )
    return row

metrics_summary = pd.DataFrame([
    score_candidate(label, candidate) for label, candidate in candidates.items()
])
metrics_summary


In [ ]:
# Compact qualitative check of the primary scoring candidate.
primary_candidate = trimmed_token_capped_generation
word_comparison = pd.DataFrame({
    'position': range(1, min(len(primary_candidate.split()), target_word_count) + 1),
    'generated': primary_candidate.split(),
    'target': target_words[:len(primary_candidate.split())],
})
word_comparison['exact_word_match'] = (
    word_comparison['generated'].str.lower() == word_comparison['target'].str.lower()
)
word_comparison


### Method note

The main evaluation uses `trimmed_token_capped_generation`. It uses a generous model-level token cap, then applies deterministic word-level trimming so every metric compares the same known-length continuation. Keep the other candidates as length-sensitivity diagnostics.